# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [35]:
! pip install -q schedule pytest # установка библиотек, если ещё не

3909.20s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [4]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [5]:
import re
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'

def get_book_data(book_url: str) -> dict:
   """Собирает информацию книге.

    :param book_url: url книги
    :type book_url: str
    :return: Словарь, заполненый доступными данными о книге (если что-то не удалось считать, в выводе это включено не будет)
    :rtype: dict

    """

   # НАЧАЛО ВАШЕГО РЕШЕНИЯ
   response = requests.get(book_url)
   response.raise_for_status()
   html_doc = response.content
   soup = BeautifulSoup(html_doc, 'html.parser')
   soup.prettify()
   book_data = {}
   product_main_data = soup.find('div', class_='product_main')
   book_name = product_main_data.find('h1').get_text()
   book_price = product_main_data.find('p', class_='price_color').get_text()
   book_data['name'] = book_name
   book_data['price'] = book_price

   book_available_tag = product_main_data.find('p', attrs ={'class':['instock', 'availability']})
   if book_available_tag.find('i', class_ = 'icon-ok') != None:
      available_text = book_available_tag.get_text().strip('\n ')
      parsed_available_books_number = int(re.search(r'.+\((\d+) available\)', available_text).group(1))
      book_data['available'] = parsed_available_books_number
   else:
      book_data['available'] = 0

   book_rating_item_rating_classes = product_main_data.find('p', attrs = {'class': 'star-rating'}).attrs.get('class')
   book_rating = None
   for _class in book_rating_item_rating_classes:
      if _class == 'One':
         book_rating = 1
         break
      if _class == 'Two':
         book_rating = 2
         break
      if _class == 'Three':
         book_rating = 3
         break
      if _class == 'Four':
         book_rating = 4
      if _class == 'Five':
         book_rating = 5
         break

   if book_rating is not None:
      book_data['rating'] = book_rating
      
   try:
      product_description = soup.find('div', attrs = {'id':'product_description'}).find_next('p').get_text()
      book_data['product_description'] = product_description
   except:
      pass
      #ignore

   product_additional_data_table_items = (soup.find('article', class_ = 'product_page')
                                       .find('table', attrs = {'class': ['table','table-striped']}))

   for item in product_additional_data_table_items:
      if item == '\n':
         continue
      name = item.find('th').get_text()
      value = item.find('td').get_text()
      book_data['additional_parameter_'+ name] = value

   return book_data
   
   # КОНЕЦ ВАШЕГО РЕШЕНИЯ


In [6]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'name': 'A Light in the Attic',
 'price': '£51.77',
 'available': 22,
 'rating': 3,
 'product_description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [7]:
catalog_url = 'https://books.toscrape.com/catalogue'

def scrape_books(catalog_url, is_save=False):
   
    """Собирает информацию о книгах в каталоге.

    :param catalog_url: url каталога книг
    :type catalog_url: str
    :param is_save: сохранить вывод в файл
    :type is_save: bool
    :return: Лист словарей, заполненых данными о книгах
    :rtype: list

    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    books_catalog_url = catalog_url
    result = []
    page_number = 1
    while(True):
        books_pages_url = books_catalog_url + '/category/books_1' + ('/page-'+ str(page_number) if page_number > 1 else '/index') + '.html'
        response = requests.get(books_pages_url,timeout=180)
        if response.status_code == 404:
            break

        response.raise_for_status()
        html_doc = response.content
        soup = BeautifulSoup(html_doc, 'html.parser')
        all_books_on_page = soup.find_all('article', class_ = 'product_pod')

        books_href = []
        for book_item in all_books_on_page:
            href_to_book_page = book_item.find('div', class_ = 'image_container').find('a')
            if href_to_book_page.has_attr('href') and len(href_to_book_page.attrs.get('href')) > 0:
                href = href_to_book_page.attrs.get('href')
                href = href.split('..')[-1]
                books_href.append(books_catalog_url + href)
        
        for url in books_href:
            book_data = get_book_data(url)
            result.append(book_data)
        
        page_number = page_number + 1
    
    if is_save:
        with open('books_data.txt', 'w') as file:
            for book_info in result:
                file.write(str(book_info)+'\n')

    return result
 
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [9]:
# Проверка работоспособности функции
res = scrape_books('https://books.toscrape.com/catalogue', is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

ConnectTimeout: HTTPSConnectionPool(host='books.toscrape.com', port=443): Max retries exceeded with url: /catalogue/ship-leaves-harbor-essays-on-travel-by-a-recovering-journeyman_438/index.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x75652ad8f3e0>, 'Connection to books.toscrape.com timed out. (connect timeout=None)'))

## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [41]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

# КОНЕЦ ВАШЕГО РЕШЕНИЯ

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [42]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest test/test_scraper.py

4646.24s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-8.4.2, pluggy-1.6.0
rootdir: /home/ln/Desktop/python
collected 0 items                                                              

============================ no tests ran in 0.00s =============================
ERROR: file or directory not found: test/test_scraper.py



## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```